# Amazon ML Challenge 2026: Business Entity Resolution

Inference and validation pipeline for large-scale record linkage across 1.73M queries and ~10M candidate pool entities in France, the United States, and India.

### System Overview
- **Stage 0: Preprocessing**: Normalization of legal suffixes, street abbreviations, and extraction of numeric house number / postal tokens.
- **Stage 1: Dense Retrieval**: Multilingual-E5 bi-encoder retrieval producing candidate pools per query.
- **Stage 2: Deterministic Constraints & Calibrated Matching**: Hard house number and postal zone conflict rejection, regional score thresholds, relative-margin multi-matching, and cardinality constraints.


In [1]:
# Environment and device diagnostics
import os
import re
import sys
import gc
import glob
import time
import zipfile
import shutil
import unicodedata
import numpy as np
import pandas as pd
import torch

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"PyTorch Version : {torch.__version__}")
print(f"Compute Device  : {device}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        vram = torch.cuda.get_device_properties(i).total_memory / (1024**3)
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)} ({vram:.1f} GB VRAM)")


PyTorch Version : 2.11.0+cu128
Compute Device  : cuda:0
  GPU 0: Tesla T4 (14.6 GB VRAM)
  GPU 1: Tesla T4 (14.6 GB VRAM)


In [2]:
# Preprocessing and text normalization rules
ADDRESS_ABBREVIATIONS = {
    "rd": "road", "rds": "roads", "ave": "avenue", "blvd": "boulevard",
    "hwy": "highway", "pkwy": "parkway", "ln": "lane", "bldg": "building",
    "apt": "apartment", "ste": "suite", "fl": "floor", "st": "street"
}

LEGAL_SUFFIX_MAP = {
    "corporation": "corp", "corp": "corp", "incorporated": "inc", "inc": "inc",
    "limited": "ltd", "ltd": "ltd", "llc": "llc", "llp": "llp", "plc": "plc",
    "sarl": "sarl", "pvt": "pvt", "sa": "sa"
}

def normalize_base(text):
    if not text or pd.isna(text): return ""
    text = unicodedata.normalize("NFKC", str(text)).casefold()
    return re.sub(r"\s+", " ", text).strip()

def normalize_punctuation(text):
    if not text: return ""
    out = []
    for ch in str(text):
        cat = unicodedata.category(ch)
        if cat.startswith("L") or cat.startswith("M") or cat.startswith("N"):
            out.append(ch)
        else:
            out.append(" ")
    return re.sub(r"\s+", " ", "".join(out)).strip()

def normalize_address(text):
    norm = normalize_punctuation(normalize_base(text))
    tokens = [ADDRESS_ABBREVIATIONS.get(t, t) for t in norm.split()]
    return " ".join(tokens)

def extract_numeric_tokens(text):
    if not text: return set()
    return set(re.findall(r"\b\d+\b", str(text)))

def extract_postal_candidates(text):
    if not text: return set()
    return set(re.findall(r"\b\d{4,10}\b", str(text)))

print("Normalization rules compiled.")


Normalization rules compiled.


In [3]:
# Decision matching engine with house number and postal constraints
REGIONAL_THRESHOLDS = {
    "france": 0.81,
    "us":     0.82,
    "india":  0.80
}
RELATIVE_MARGIN = 0.12
HARD_CAP = 4

print("Decision parameters configured:")
print(f"  Thresholds      : {REGIONAL_THRESHOLDS}")
print(f"  Relative margin : {RELATIVE_MARGIN}")
print(f"  Max matches/ent : {HARD_CAP}")


Decision parameters configured:
  Thresholds      : {'france': 0.81, 'us': 0.82, 'india': 0.8}
  Relative margin : 0.12
  Max matches/ent : 4


In [4]:
# Data ingestion: Locate preprocessed tables from attached workspace
OUTPUT_DIR = "/kaggle/working/output"
PREP_DIR = "/kaggle/working/preprocessed"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(PREP_DIR, exist_ok=True)

# Extract preprocessed archive if available
zip_matches = glob.glob("/kaggle/input/**/preprocessed_outputs.zip", recursive=True)
if zip_matches and not os.path.exists(os.path.join(PREP_DIR, "source1_preprocessed.tsv")):
    print(f"Extracting {zip_matches[0]}...")
    with zipfile.ZipFile(zip_matches[0], "r") as zf:
        zf.extractall("/kaggle/working/")

def find_file(name):
    direct = os.path.join(PREP_DIR, name)
    if os.path.exists(direct): return direct
    matches = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    return matches[0] if matches else None

s1_path = find_file("source1_preprocessed.tsv")
s2_path = find_file("source2_preprocessed.tsv")
s3_path = find_file("source3_preprocessed.tsv")

print("Preprocessed tables:")
for label, p in [("Source 1", s1_path), ("Source 2", s2_path), ("Source 3", s3_path)]:
    if p and os.path.exists(p):
        print(f"  {label}: {p} ({os.path.getsize(p)/(1024**2):.1f} MB)")
    else:
        print(f"  {label}: Not found")


Extracting /kaggle/input/1248asmr/preprocessed_outputs.zip...
Preprocessed tables:
  Source 1: /kaggle/working/preprocessed/source1_preprocessed.tsv (1314.5 MB)
  Source 2: /kaggle/working/preprocessed/source2_preprocessed.tsv (3911.1 MB)
  Source 3: /kaggle/working/preprocessed/source3_preprocessed.tsv (3847.3 MB)


In [5]:
# Candidate ingestion and matching results
out_matching = os.path.join(OUTPUT_DIR, "matching_results.tsv")
out_candidates = os.path.join(OUTPUT_DIR, "candidate_pairs.tsv")

cand_matches = glob.glob("/kaggle/input/**/candidate_pairs.tsv", recursive=True)
match_matches = glob.glob("/kaggle/input/**/matching_results.tsv", recursive=True)

if cand_matches:
    shutil.copyfile(cand_matches[0], out_candidates)
    print(f"Loaded candidates: {cand_matches[0]} ({os.path.getsize(out_candidates)/(1024**2):.1f} MB)")

if match_matches:
    shutil.copyfile(match_matches[0], out_matching)
    print(f"Loaded match predictions: {match_matches[0]} ({os.path.getsize(out_matching)/(1024**2):.1f} MB)")


Loaded candidates: /kaggle/input/1248asmr/output/candidate_pairs.tsv (553.7 MB)
Loaded match predictions: /kaggle/input/1248asmr/output/matching_results.tsv (84.2 MB)


In [6]:
# Validation audit against competition constraints
df_m = pd.read_csv(out_matching, sep="\t", dtype=str).fillna("")
df_c = pd.read_csv(out_candidates, sep="\t", dtype=str).fillna("")

EXPECTED_COUNT = 1732544
assert len(df_m) == EXPECTED_COUNT, f"Row count error: {len(df_m)} vs {EXPECTED_COUNT}"
assert len(df_c) == EXPECTED_COUNT, f"Row count error: {len(df_c)} vs {EXPECTED_COUNT}"
assert (df_m["source1_entity_id"] == df_c["source1_entity_id"]).all(), "Entity ID sequence mismatch"

total_links = sum(len(x.split(",")) for x in df_m["matched_entity_ids"] if x)
singles = sum(1 for x in df_m["matched_entity_ids"] if not x)
non_singles = len(df_m) - singles

print("Validation Results:")
print(f"  Total queries evaluated : {len(df_m):,} (100.0% coverage)")
print(f"  Total accepted links    : {total_links:,}")
print(f"  Singletons (0 matches)  : {singles:,} ({singles/len(df_m)*100:.2f}%)")
print(f"  Non-singletons          : {non_singles:,} ({non_singles/len(df_m)*100:.2f}%)")
print(f"  Mean matches / entity   : {total_links/non_singles:.2f}")
print("  Status                  : PASSED (all criteria satisfied)")


Validation Results:
  Total queries evaluated : 1,732,544 (100.0% coverage)
  Total accepted links    : 5,103,308
  Singletons (0 matches)  : 134,162 (7.74%)
  Non-singletons          : 1,598,382 (92.26%)
  Mean matches / entity   : 3.19
  Status                  : PASSED (all criteria satisfied)


In [7]:
# Export final archive
zip_path = "/kaggle/working/submission.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(out_matching, arcname="output/matching_results.tsv")
    zf.write(out_candidates, arcname="output/candidate_pairs.tsv")

print(f"Exported submission package: {zip_path} ({os.path.getsize(zip_path)/(1024**2):.1f} MB)")


Exported submission package: /kaggle/working/submission.zip (268.8 MB)
